In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(2*np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)))

def eval_metrics(name, y_true, y_pred):
    return {
        "model": name,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(root_mean_squared_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "SMAPE": float(smape(y_true, y_pred)),
    }

def split_by_day(df, test_days=7, val_days=7):
    max_day = int(df["d"].max())
    test_start = max_day - test_days + 1
    val_start  = test_start - val_days

    train_df = df[df["d"] < val_start].copy()
    val_df   = df[(df["d"] >= val_start) & (df["d"] < test_start)].copy()
    test_df  = df[df["d"] >= test_start].copy()

    return train_df, val_df, test_df

# 補零
def densify_topk_series(df_raw: pd.DataFrame, top_k=20000):
    """
    df_raw 需要欄位: d,t,x,y,count
    回傳：只含 top_k 個 (x,y,t) 且已補齊所有 d 的 DataFrame
    """
    df = df_raw[["d","t","x","y","count"]].copy()

    # 選最活躍的 (x,y,t)：用總量或出現天數都可以
    key_sum = df.groupby(["x","y","t"])["count"].sum().sort_values(ascending=False)
    top_keys = key_sum.head(top_k).index

    df = df.set_index(["x","y","t"]).loc[top_keys].reset_index()

    dmin, dmax = int(df["d"].min()), int(df["d"].max())
    all_d = np.arange(dmin, dmax + 1, dtype=int)

    out = []
    for (x,y,t), g in df.groupby(["x","y","t"], sort=False):
        g2 = g.set_index("d").reindex(all_d)
        g2["count"] = g2["count"].fillna(0.0)
        g2["d"] = all_d
        g2["x"] = x; g2["y"] = y; g2["t"] = t
        out.append(g2[["d","t","x","y","count"]])

    return pd.concat(out, ignore_index=True)

In [ ]:
df = pd.read_parquet("../data/processed/sapporo_density.parquet")
df = df[~((df["x"]==999) & (df["y"]==999))].copy()

df_raw = df.copy()

# 只補 top_k，先用小一點，穩了再加
#df_dense = densify_topk_series(df_raw, top_k=500000)
df_dense = pd.read_parquet("../data/processed/sapporo_density_filled_zero.parquet").copy()
print(df_dense.shape, df_dense["count"].mean(), (df_dense["count"]==0).mean())

In [ ]:
# LSTM 序列長度
SEQ_LEN = 28

df = df_dense.copy() #如果要看沒補零的效能這行可以註解起來

# 假設 d=0 是 2023-01-01）
df["date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(df["d"], unit="D")
df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

# 排序
df = df.sort_values(["x","y","t","d"])

# 產生 lag_1..lag_SEQ_LEN（給 LSTM 當序列，也給 baseline）
g = df.groupby(["x","y","t"])["count"]
for k in range(1, SEQ_LEN+1):
    df[f"lag_{k}"] = g.shift(k)

# rolling 特徵（給 LGBM 用）
df["rolling_3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
df["rolling_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())

# 丟掉 lag 不足的列
need_cols = [f"lag_{k}" for k in range(1, SEQ_LEN+1)] + ["rolling_3","rolling_7"]
df = df.dropna(subset=need_cols).copy()

train_df, val_df, test_df = split_by_day(df, test_days=7, val_days=7)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
def baseline_lagk(df_split, k=7):
    return df_split[f"lag_{k}"].to_numpy()

def fit_baseline_hist(train_df):
    # (weekday,t,x,y) 的歷史平均
    hist = train_df.groupby(["weekday","t","x","y"])["count"].mean()
    return hist

def predict_baseline_hist(df_split, hist_series):
    key = list(zip(df_split["weekday"], df_split["t"], df_split["x"], df_split["y"]))
    # 沒看過的 key 用全域平均補
    global_mean = float(hist_series.mean())
    pred = np.array([hist_series.get(k, global_mean) for k in key], dtype=float)
    return pred

In [ ]:
import lightgbm as lgb

LGB_FEATURES = ["weekday","t","x","y","is_weekend","lag_1","lag_7","rolling_3","rolling_7"]

def train_lgbm(train_df, val_df, sample_n=800_000, seed=42):
    tr = train_df.sample(n=min(sample_n, len(train_df)), random_state=seed)
    X_tr, y_tr = tr[LGB_FEATURES], tr["count"]
    X_va, y_va = val_df[LGB_FEATURES], val_df["count"]

    model = lgb.LGBMRegressor(
        n_estimators=4000,
        learning_rate=0.05,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=4
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50)]
    )
    return model

def predict_lgbm(model, df_split):
    return model.predict(df_split[LGB_FEATURES])

In [ ]:
import torch, pickle

def save_lstm_pkl(model, path_pkl, config: dict):
    payload = {
        "state_dict": model.state_dict(),
        "config": config
    }
    torch.save(payload, path_pkl)   # 用 torch 的序列化最穩（副檔名可叫 .pkl）
    # 或你也可以用 pickle.dump(payload, open(...,"wb"))，但 torch.save 更常用

config = {
    "seq_len": SEQ_LEN,
    "hidden": 64,
    "layers": 1,
    "lr": 1e-3,

    "n_weekday": 7,
    "n_t": 48,
    "n_x": int(df["x"].max()) + 1,
    "n_y": int(df["y"].max()) + 1,

    "emb_wd": 2,
    "emb_t": 8,
    "emb_x": 16,
    "emb_y": 16,
}



In [ ]:
from src.LSTM import *

def run_all_models(train_df, val_df, test_df, seq_len=14):
    results = []

    # ---- baselines ----
    hist = fit_baseline_hist(train_df)

    for _df in [train_df, val_df, test_df]:
        _df["y_hist"] = predict_baseline_hist(_df, hist)              # baseline in count scale
        _df["base_log"] = np.log1p(_df["y_hist"].to_numpy())          # log1p baseline
        _df["y_target"] = np.log1p(_df["count"].to_numpy()) - _df["base_log"]  # residual in log space

    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()

        yhat = predict_baseline_hist(df_split, hist)
        results.append(eval_metrics(f"baseline_hist ({split_name})", y, yhat))

        yhat = baseline_lagk(df_split, k=1)
        results.append(eval_metrics(f"baseline_lag1 ({split_name})", y, yhat))

        yhat = baseline_lagk(df_split, k=7)
        results.append(eval_metrics(f"baseline_lag7 ({split_name})", y, yhat))

    # ---- LGBM ----
    lgbm = train_lgbm(train_df, val_df)
    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()
        yhat = predict_lgbm(lgbm, df_split)
        results.append(eval_metrics(f"LGBM ({split_name})", y, yhat))

    # ---- LSTM ----
    spec = {
        "hidden": 64,
        "layers": 1,
        "n_weekday": 7,
        "n_t": 48,
        "n_x": int(pd.concat([train_df["x"], val_df["x"]]).max()) + 1,
        "n_y": int(pd.concat([train_df["y"], val_df["y"]]).max()) + 1,
        "emb_wd": 2,
        "emb_t": 8,
        "emb_x": 16,
        "emb_y": 16,
        "seq_len": SEQ_LEN,
        "epochs": 20,
        "lr": 1e-3,
        "batch_size": 2048,
        "seed": 42,
    }

    model = LSTMRegEmbed(
        hidden=spec["hidden"],
        layers=spec["layers"],
        n_weekday=spec["n_weekday"],
        n_t=spec["n_t"],
        n_x=spec["n_x"],
        n_y=spec["n_y"],
        emb_wd=spec["emb_wd"],
        emb_t=spec["emb_t"],
        emb_x=spec["emb_x"],
        emb_y=spec["emb_y"],
    )

    lstm = train_lstm_embed(
        model=model,
        train_df=train_df,
        val_df=val_df,
        seq_len=spec["seq_len"],
        epochs=spec.get("epochs", 20),
        lr=spec.get("lr", 1e-3),
        batch_size=spec.get("batch_size", 1024),
        seed=spec.get("seed", 42),
        loss="huber", 
        huber_beta=2.0,
        use_residual=True
    ) 

    save_lstm_pkl(lstm, "../models/lstm_sapporo.pkl", config) #存成PKL檔案

    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()
        yhat = predict_lstm_embed(lstm, df_split, seq_len=seq_len, use_residual=True)
        results.append(eval_metrics(f"LSTM ({split_name})", y, yhat))

    return pd.DataFrame(results).sort_values(["model"]).reset_index(drop=True)

report = run_all_models(train_df, val_df, test_df, seq_len=SEQ_LEN)
report